# Training Data — EDA, Preprocessing & Model
Load training data → inspect → clean → feature engineering → train Linear Regression → evaluate

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

## Step 1 — Load & Inspect Training Data

In [ ]:
df_train = pd.read_csv('../data/training.csv')
print('Shape:', df_train.shape)
print('Columns:', df_train.columns.tolist())
print('\nDtypes:\n', df_train.dtypes)
print('\nMissing values:\n', df_train.isna().sum())
df_train.head()

In [ ]:
# Inspect categorical and binary columns
print('state_holiday unique:', df_train['state_holiday'].unique())
print('open unique:         ', df_train['open'].unique())
print('Closed rows:         ', len(df_train[df_train['open'] == 0]))
print('Open rows:           ', len(df_train[df_train['open'] == 1]))

## Step 2 — EDA: Anomaly Detection
Check logical edge cases:
- `sales = 0` and `customers = 0` — expected (closed store)
- `sales = 0` but `customers > 0` — anomaly (open store, customers present, zero sales)

In [ ]:
# When sales=0: are customers always 0?
print('=== When sales == 0 ===')
print(df_train[df_train['sales'] == 0]['nb_customers_on_day'].value_counts())

# When customers=0: are sales always 0?
print('\n=== When customers == 0 ===')
print(df_train[df_train['nb_customers_on_day'] == 0]['sales'].value_counts())

In [ ]:
# Find the anomalous row: open store + customers > 0 but sales = 0
anomaly = df_train[(df_train['sales'] == 0) & (df_train['nb_customers_on_day'] > 0)]
print('Anomalous rows found:', len(anomaly))
print(anomaly)

In [ ]:
# Quantify the anomaly as a percentage
# Filter: open stores with customers
open_with_customers = df_train[(df_train['open'] == 1) & (df_train['nb_customers_on_day'] > 0)]

# Count total and anomalies
total     = len(open_with_customers)
anomalies = len(open_with_customers[open_with_customers['sales'] == 0])

# Calculate percentage
pct = (anomalies / total) * 100

print(f'Total open days with customers: {total}')
print(f'Days with sales = 0:            {anomalies}')
print(f'Anomaly percentage:             {pct:.4f}%')

## Step 3 — Data Cleaning

In [ ]:
# Drop index column — just a row identifier, not a feature
df_train = df_train.drop(columns=['Unnamed: 0'])
print('Dropped index column. New shape:', df_train.shape)

In [ ]:
# Drop the 1 anomalous row: open store + customers > 0 + sales = 0
# Reason: 0.0002% of data — data entry error, contradicts the learned pattern
anomaly_mask = (df_train['open'] == 1) & (df_train['nb_customers_on_day'] > 0) & (df_train['sales'] == 0)
df_train = df_train[~anomaly_mask]
print(f'Removed {anomaly_mask.sum()} anomalous row(s). Shape: {df_train.shape}')

In [ ]:
# Keep only open stores — closed stores always have sales=0
# They don't teach the model anything about what drives sales
# We will handle closed stores separately at prediction time (force sales=0)
df_train = df_train[df_train['open'] == 1].copy()
print(f'Rows after removing closed stores: {len(df_train)}')

## Step 4 — Feature Engineering

In [ ]:
# Fix date column type: string → datetime → extract numeric features
# Training data format: YYYY-MM-DD
df_train['date']       = pd.to_datetime(df_train['date'])
df_train['year']       = df_train['date'].dt.year         # captures year-over-year trends
df_train['month']      = df_train['date'].dt.month        # captures seasonality (Dec = high sales)
df_train['day']        = df_train['date'].dt.day          # captures month-end effects
df_train['week']       = df_train['date'].dt.isocalendar().week.astype(int)  # weekly patterns
df_train['is_weekend'] = df_train['date'].dt.dayofweek.isin([5, 6]).astype(int)  # weekend flag
df_train = df_train.drop(columns=['date'])  # drop original string column — no longer needed
print('Date features extracted. Shape:', df_train.shape)

In [ ]:
# Encode state_holiday — One-Hot Encoding (required for Linear Regression)
# Values: '0'=no holiday, 'a'=public, 'b'=easter, 'c'=christmas
# drop_first=True: drops holiday_0 to avoid dummy variable trap (multicollinearity)
# Must cast to str BEFORE get_dummies to ensure consistent type
df_train['state_holiday'] = df_train['state_holiday'].astype(str)
df_train = pd.get_dummies(df_train, columns=['state_holiday'], prefix='holiday', drop_first=True)

# Convert boolean OHE columns to int (0/1) for model compatibility
for col in ['holiday_a', 'holiday_b', 'holiday_c']:
    if col in df_train.columns:
        df_train[col] = df_train[col].astype(int)

print('Encoding done. Columns:', df_train.columns.tolist())
df_train.head()

## Step 5 — Define Features & Target

In [ ]:
# X = all features (everything except what we want to predict)
# y = target variable (what we want to predict)
X = df_train.drop(columns=['sales'])
y = df_train['sales']

# Save feature columns — needed to align REAL_DATA later
feature_columns = X.columns.tolist()
print('Features:', feature_columns)
print('Target: sales')
print('X shape:', X.shape, '| y shape:', y.shape)

## Step 6 — Train / Validation Split

In [ ]:
# Split into training (75%) and validation (25%) sets
# random_state=42 ensures reproducibility (same split every run)
# Validation set simulates how the model performs on unseen data
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print(f'Train size:      {len(X_train)}')
print(f'Validation size: {len(X_val)}')

## Step 7 — Train Linear Regression

In [ ]:
# Train the model — LinearRegression finds the best-fit line through the data
# It learns the weight (coefficient) of each feature
lr = LinearRegression()
lr.fit(X_train, y_train)
print('Model trained.')

## Step 8 — Evaluate

In [ ]:
# R2 Train      — how well the model learned the training data
# R2 Validation — how much we expect the model to perform on unseen data (our prediction)
# RMSE          — average error in euros (lower = better)
# R2 Difference — gap between train and validation (close to 0 = good generalisation)

y_train_pred = lr.predict(X_train)
y_val_pred   = lr.predict(X_val)

r2_train      = r2_score(y_train, y_train_pred)
r2_validation = r2_score(y_val,   y_val_pred)
rmse          = np.sqrt(mean_squared_error(y_val, y_val_pred))
r2_difference = r2_train - r2_validation

print('=== Linear Regression Results ===')
print(f'R2 Train      : {r2_train:.4f}  ← how well model learned training data')
print(f'R2 Validation : {r2_validation:.4f}  ← expected performance on real data')
print(f'RMSE          : {rmse:.2f}  ← average error in euros')
print(f'R2 Difference : {r2_difference:.4f}  ← gap (closer to 0 = better generalisation)')

## Step 9 — Feature Importance

In [ ]:
# Coefficients show how much each feature influences the prediction
# Positive = increases sales | Negative = decreases sales
coefficients = pd.DataFrame({
    'Feature'    : X.columns,
    'Coefficient': lr.coef_
}).sort_values('Coefficient', ascending=False)

print('=== Feature Importance (Linear Regression Coefficients) ===')
print(coefficients.to_string(index=False))